<a href="https://colab.research.google.com/github/Sashanka2001/Image_Quality_Feature_Extraction_Colab/blob/main/02_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1: MOUNT DRIVE & DEPENDENCY INSTALLATION

import os
from google.colab import drive

# Mount Google Drive only if it isn't mounted yet
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("⚡ Google Drive is already mounted!")

# Install dependencies quietly
!pip install opencv-python pandas tqdm scikit-image scipy openpyxl -q

print("\n✅ Google Drive mounted & environment ready!")

Mounted at /content/drive

✅ Google Drive mounted & environment ready!


In [ ]:
# CELL 2: 34-FEATURE EXTRACTION ENGINE WITH CHECKPOINT RESUME

import os
import cv2
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.measure import shannon_entropy
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import skew, kurtosis

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

DRIVE_PROJECT = "/content/drive/MyDrive/pre_enhancement_facial_recoverability"
DIR_METADATA = os.path.join(DRIVE_PROJECT, "dataset/metadata")
LOCAL_METADATA_DIR = "dataset/metadata"

METADATA_CSV_PATH = os.path.join(DIR_METADATA, "metadata.csv")
LOCAL_METADATA_CSV = os.path.join(LOCAL_METADATA_DIR, "metadata.csv")

DRIVE_FEATURE_CSV = os.path.join(DIR_METADATA, "image_quality_features.csv")
LOCAL_FEATURE_CSV = os.path.join(LOCAL_METADATA_DIR, "image_quality_features.csv")

DRIVE_FEATURE_XLSX = os.path.join(DIR_METADATA, "image_quality_features.xlsx")
LOCAL_FEATURE_XLSX = os.path.join(LOCAL_METADATA_DIR, "image_quality_features.xlsx")

CHECKPOINT_INTERVAL = 1000  # Syncs progress to Google Drive every 1,000 images

# ------------------------------------------------------------
# 2. FEATURE EXTRACTION METRICS (34 FEATURES / 8 DOMAINS)
# ------------------------------------------------------------

def safe_val(val):
    if np.isnan(val) or np.isinf(val):
        return 0.0
    return float(val)

def extract_all_quality_metrics(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # 1. Sharpness (5 features)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)

    sharpness_feats = {
        "laplacian_variance": safe_val(np.var(laplacian)),
        "gradient_mean": safe_val(np.mean(grad_mag)),
        "gradient_std": safe_val(np.std(grad_mag)),
        "tenengrad": safe_val(np.mean(sobel_x ** 2 + sobel_y ** 2)),
        "high_frequency_energy": safe_val(np.mean(laplacian ** 2))
    }

    # 2. Noise (2 features)
    smoothed = cv2.GaussianBlur(gray, (3, 3), 0)
    residual = gray.astype(np.float32) - smoothed.astype(np.float32)
    noise_feats = {
        "estimated_noise_std": safe_val(np.std(residual)),
        "estimated_noise_variance": safe_val(np.var(residual))
    }

    # 3. Brightness & Distribution (8 features)
    brightness_feats = {
        "mean_intensity": safe_val(np.mean(gray)),
        "median_intensity": safe_val(np.median(gray)),
        "min_intensity": safe_val(np.min(gray)),
        "max_intensity": safe_val(np.max(gray)),
        "underexposed_ratio": safe_val(np.mean(gray < 30)),
        "overexposed_ratio": safe_val(np.mean(gray > 225)),
        "intensity_skewness": safe_val(skew(gray.flatten().astype(np.float32))),
        "intensity_kurtosis": safe_val(kurtosis(gray.flatten().astype(np.float32)))
    }

    # 4. Contrast (3 features)
    p5, p95 = np.percentile(gray, 5), np.percentile(gray, 95)
    contrast_feats = {
        "rms_contrast": safe_val(np.sqrt(np.mean((gray - np.mean(gray)) ** 2))),
        "intensity_std": safe_val(np.std(gray)),
        "percentile_contrast": safe_val(p95 - p5)
    }

    # 5. Entropy (1 feature)
    entropy_feats = {
        "image_entropy": safe_val(shannon_entropy(gray))
    }

    # 6. Edge Density & Area (2 features)
    edges = cv2.Canny(gray, threshold1=100, threshold2=200)
    h, w = img.shape[:2]
    edge_feats = {
        "edge_density": safe_val(np.mean(edges > 0)),
        "image_area": int(w * h)
    }

    # 7. Texture GLCM (4 features)
    quantized = (gray / 32).astype(np.uint8)
    glcm = graycomatrix(quantized, distances=[1], angles=[0], levels=8, symmetric=True, normed=True)
    texture_feats = {
        "texture_contrast": safe_val(graycoprops(glcm, "contrast")[0, 0]),
        "texture_homogeneity": safe_val(graycoprops(glcm, "homogeneity")[0, 0]),
        "texture_energy": safe_val(graycoprops(glcm, "energy")[0, 0]),
        "texture_correlation": safe_val(graycoprops(glcm, "correlation")[0, 0])
    }

    # 8. Color Metrics (9 features)
    color_feats = {
        "mean_blue": safe_val(np.mean(img[:, :, 0])),
        "mean_green": safe_val(np.mean(img[:, :, 1])),
        "mean_red": safe_val(np.mean(img[:, :, 2])),
        "mean_hue": safe_val(np.mean(hsv[:, :, 0])),
        "mean_saturation": safe_val(np.mean(hsv[:, :, 1])),
        "mean_value": safe_val(np.mean(hsv[:, :, 2])),
        "mean_lab_l": safe_val(np.mean(lab[:, :, 0])),
        "mean_lab_a": safe_val(np.mean(lab[:, :, 1])),
        "mean_lab_b": safe_val(np.mean(lab[:, :, 2]))
    }

    # Merge all 34 features
    all_metrics = {}
    for sub_dict in [sharpness_feats, noise_feats, brightness_feats, contrast_feats,
                     entropy_feats, edge_feats, texture_feats, color_feats]:
        all_metrics.update(sub_dict)

    return all_metrics

# ------------------------------------------------------------
# 3. SAVE & EXPORT HELPER (CSV + EXCEL DUAL SYNC)
# ------------------------------------------------------------

def save_and_sync_datasets(feature_df):
    feature_df.to_csv(DRIVE_FEATURE_CSV, index=False)
    feature_df.to_csv(LOCAL_FEATURE_CSV, index=False)

    with pd.ExcelWriter(DRIVE_FEATURE_XLSX, engine='openpyxl') as writer:
        feature_df.to_excel(writer, index=False, sheet_name='Quality_Features')
        worksheet = writer.sheets['Quality_Features']

        for col in worksheet.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = col[0].column_letter
            worksheet.column_dimensions[col_letter].width = max(max_len + 3, 14)

    shutil.copy(DRIVE_FEATURE_XLSX, LOCAL_FEATURE_XLSX)

# ------------------------------------------------------------
# 4. EXECUTION PIPELINE WITH AUTOMATIC RESUME
# ------------------------------------------------------------

def run_feature_extraction():
    os.makedirs(DIR_METADATA, exist_ok=True)
    os.makedirs(LOCAL_METADATA_DIR, exist_ok=True)

    if os.path.exists(METADATA_CSV_PATH):
        metadata = pd.read_csv(METADATA_CSV_PATH)
    elif os.path.exists(LOCAL_METADATA_CSV):
        metadata = pd.read_csv(LOCAL_METADATA_CSV)
    else:
        raise FileNotFoundError("Could not find 'metadata.csv'. Run Notebook 1 first.")

    total_images = len(metadata)
    print(f"📊 Total degraded images found: {total_images:,}")

    processed_files = set()
    feature_records = []

    if os.path.exists(DRIVE_FEATURE_CSV):
        existing_df = pd.read_csv(DRIVE_FEATURE_CSV)
        processed_files = set(existing_df["degraded_filename"].tolist())
        feature_records = existing_df.to_dict('records')
        print(f"🔄 Resuming session: Skipping {len(processed_files):,} previously processed images.")
    else:
        print("🚀 Starting fresh feature extraction session...")

    new_processed_count = 0

    for idx, row in tqdm(metadata.iterrows(), total=total_images, desc="Extracting Features"):
        deg_filename = row["degraded_filename"]

        if deg_filename in processed_files:
            continue

        image_path = row["drive_path"]
        metrics = extract_all_quality_metrics(image_path)

        if metrics is None:
            continue

        record = {
            "original_filename": row["original_filename"],
            "cropped_filename": row["cropped_filename"],
            "degraded_filename": deg_filename,
            "degradation_type": row["degradation_type"],
            "severity_level": row["severity_level"],
            "severity_parameter": row["severity_parameter"],
            "parameter_value": row["parameter_value"],
            "drive_path": image_path
        }
        record.update(metrics)
        feature_records.append(record)
        processed_files.add(deg_filename)
        new_processed_count += 1

        if new_processed_count % CHECKPOINT_INTERVAL == 0:
            temp_df = pd.DataFrame(feature_records)
            save_and_sync_datasets(temp_df)
            print(f"\n💾 Checkpoint saved at {len(feature_records):,} total images.")

    final_df = pd.DataFrame(feature_records)
    save_and_sync_datasets(final_df)

    print("\n--------------------------------------------------")
    print("✅ FEATURE EXTRACTION COMPLETE")
    print(f"📊 Total Images Processed : {len(final_df):,}")
    print(f"📐 Total Dataset Columns   : {len(final_df.columns)}")
    print(f"☁️ Google Drive CSV Path   : '{DRIVE_FEATURE_CSV}'")
    print(f"☁️ Google Drive XLSX Path  : '{DRIVE_FEATURE_XLSX}'")
    print(f"💻 Local Colab Disk Path    : '{LOCAL_FEATURE_CSV}'")

    return final_df

feature_df = run_feature_extraction()

📊 Total degraded images found: 56,000
🔄 Resuming session: Skipping 34,700 previously processed images.


Extracting Features:  62%|██████▏   | 34444/56000 [00:19<00:04, 5363.61it/s]